# 04 — Demographics Pipeline

## Purpose
Build a rich demographic profile of aged care users at ACPR level.
Output is intentionally kept granular so any downstream analysis can slice freely.

## Input
- `data/raw/admission/` — admission CURFs 2019-20 → 2023-24
- `data/raw/people_using_aged_care/` — people using care 2018-19 → 2024

## Output
- `data/clean/demographics_acpr.csv` — summary per ACPR × year × care_type
  with participation rate, NESB rate, Indigenous rate, HCP level distribution

## Key metrics
- **Participation rate** = pct_first_admission + pct_repeat_admission (from FIRST_ADMISSION col)
- **NESB** = Country of Birth is Non-English-speaking country
- **HCP level** = which package level is allocated most per ACPR
- **Indigenous** = Aboriginal or Torres Strait Islander status

## Data caveat
- All files are at **ACPR level** (73 regions) — no SA3 available in CURFs
- Residential files 2019-21 have `Admission_type` (Permanent/Respite) instead of FIRST_ADMISSION Yes/No
- `–` and `-` in FIRST_ADMISSION = not stated, treated as NaN

In [ ]:
import pandas as pd
import numpy as np
import os

RAW_ADM    = '../../data/raw/admission'
RAW_PEOPLE = '../../data/raw/people_using_aged_care'
OUT        = '../../data/clean/demographics_acpr.csv'

In [ ]:
# =============================================================================
# STEP 1: Load and combine all admission + people_using files
# =============================================================================

def load_file(path):
    if path.endswith('.xlsx'):
        return pd.read_excel(path)
    try:
        return pd.read_csv(path, encoding='utf-8', low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding='latin1', low_memory=False)

frames = []
for folder, source in [(RAW_ADM, 'admission'), (RAW_PEOPLE, 'people_using')]:
    for fname in sorted(os.listdir(folder)):
        if fname.startswith('~'): continue
        if not (fname.endswith('.xlsx') or fname.endswith('.csv')): continue
        df = load_file(f'{folder}/{fname}')
        df['source'] = source
        df['source_file'] = fname
        frames.append(df)
        print(f'[{source}] {fname}: {df.shape}')

raw = pd.concat(frames, ignore_index=True)
print(f'\nCombined raw: {raw.shape}')
print('Columns:', raw.columns.tolist())

In [ ]:
# =============================================================================
# STEP 2: Standardise column names across all files and years
# =============================================================================

col_map = {
    # Geography
    'ACPR_code'          : 'acpr_code',
    'ACPR_CODE'          : 'acpr_code',
    'ACPR_CODE_2018'     : 'acpr_code',
    'ACPR_name'          : 'acpr_name',
    'ACPR_NAME'          : 'acpr_name',
    'ACPR_NAME_2018'     : 'acpr_name',
    'State'              : 'state',
    'STATE'              : 'state',
    # Time
    'Year'               : 'year',
    'YEAR'               : 'year',
    # Care type
    'Care_type'          : 'care_type',
    'CARE_TYPE'          : 'care_type',
    # Admission status: first time ever (Yes/No) — keep separate from Permanent/Respite
    'First_admission'    : 'first_admission',
    'First _admission'   : 'first_admission',   # note trailing space in some files
    'FIRST_ADMISSION'    : 'first_admission',
    # Admission_type / ADMTYPE = Permanent vs Respite — DIFFERENT concept, do NOT merge with first_admission
    'Admission_type'     : 'admission_type',
    'ADMTYPE'            : 'admission_type',
    # HCP level
    'Home_care_level'    : 'hcp_level',
    'HOME_CARE_LEVEL'    : 'hcp_level',
    # Demographics
    'Age_group'          : 'age_group',
    'AGE_GROUP'          : 'age_group',
    'AGE_GROUP_5'        : 'age_group',
    'Sex'                : 'sex',
    'SEX'                : 'sex',
    'Indigenous_status'  : 'indigenous_status',
    'INDIGENOUS_STATUS'  : 'indigenous_status',
    'Preferred_language' : 'preferred_language',
    'PREFERRED_LANGUAGE' : 'preferred_language',
    'Country_of_birth'   : 'country_of_birth',
    'COUNTRY_OF_BIRTH'   : 'country_of_birth',
    'COB'                : 'country_of_birth',
    'LAN'                : 'preferred_language',
    # Count (2018-19 pre-aggregated file)
    'count'              : 'record_count',
}
raw = raw.rename(columns={k: v for k, v in col_map.items() if k in raw.columns})

# After concat across files, some columns may appear twice (e.g. two source files both
# renamed to the same target). Consolidate: take first non-null value across duplicates.
dupes = raw.columns[raw.columns.duplicated()].unique()
for col in dupes:
    dup_block = raw.loc[:, raw.columns == col]
    raw = raw.loc[:, raw.columns != col]
    raw[col] = dup_block.bfill(axis=1).iloc[:, 0]

# Normalise FIRST_ADMISSION to Yes / No only — dashes and other values become NaN
if 'first_admission' in raw.columns:
    raw['first_admission'] = raw['first_admission'].astype(str).str.strip()
    raw['first_admission'] = raw['first_admission'].replace({
        '-': np.nan, '–': np.nan, 'nan': np.nan,
    })
    raw.loc[~raw['first_admission'].isin(['Yes', 'No']), 'first_admission'] = np.nan

# Normalise year to int
raw['year'] = pd.to_numeric(raw['year'], errors='coerce')
raw = raw.dropna(subset=['year'])
raw['year'] = raw['year'].astype(int)

# NESB flag: born in a non-English-speaking country
if 'country_of_birth' in raw.columns:
    raw['is_nesb'] = raw['country_of_birth'].str.lower().str.contains(
        'non-english', na=False).astype(int)

# Indigenous flag: Aboriginal or Torres Strait Islander (exclude "Non-Indigenous")
if 'indigenous_status' in raw.columns:
    raw['is_indigenous'] = (
        raw['indigenous_status'].astype(str).str.lower()
        .str.contains('aboriginal|torres strait', na=False) &
        ~raw['indigenous_status'].astype(str).str.lower().str.contains('non-', na=False)
    ).astype(int)

print('Years available:', sorted(raw['year'].unique()))
print('Care types:', raw['care_type'].dropna().unique() if 'care_type' in raw.columns else 'N/A')
print('Sources:', raw['source'].value_counts().to_dict())
print('first_admission values:', raw['first_admission'].value_counts(dropna=False).to_dict() if 'first_admission' in raw.columns else 'N/A')

In [ ]:
# =============================================================================
# STEP 3: Aggregate to ACPR × year × care_type × source
# =============================================================================

group_cols = ['acpr_code', 'acpr_name', 'state', 'year', 'care_type', 'source']
group_cols = [c for c in group_cols if c in raw.columns]

# 2018-19 file is pre-aggregated with a 'record_count' column.
# All other files: each row = 1 person, so record_count = 1.
if 'record_count' not in raw.columns:
    raw['record_count'] = 1
else:
    raw['record_count'] = pd.to_numeric(raw['record_count'], errors='coerce').fillna(1)

# Build agg dict dynamically so we don't error on missing columns
agg_dict = {
    'total_users'        : ('record_count', 'sum'),
    'n_first_admission'  : ('first_admission', lambda x: (x == 'Yes').sum()),
    'n_repeat_admission' : ('first_admission', lambda x: (x == 'No').sum()),
    'n_female'           : ('sex', lambda x: x.astype(str).str.lower().str.contains('f|female|2', na=False).sum()),
    'n_sex_stated'       : ('sex', lambda x: x.notna().sum()),
}
if 'is_nesb' in raw.columns:
    agg_dict['n_nesb'] = ('is_nesb', 'sum')
if 'is_indigenous' in raw.columns:
    agg_dict['n_indigenous'] = ('is_indigenous', 'sum')

summary = (
    raw.dropna(subset=['acpr_code'])
    .groupby(group_cols)
    .agg(**agg_dict)
    .reset_index()
)

# Participation rate denominator = total_users (includes not-stated rows)
# This gives: "out of everyone using care, X% are first-time entrants"
summary['pct_first_admission']  = summary['n_first_admission']  / summary['total_users']
summary['pct_repeat_admission'] = summary['n_repeat_admission'] / summary['total_users']

if 'n_nesb' in summary.columns:
    summary['pct_nesb'] = summary['n_nesb'] / summary['total_users']
if 'n_indigenous' in summary.columns:
    summary['pct_indigenous'] = summary['n_indigenous'] / summary['total_users']
summary['pct_female'] = summary['n_female'] / summary['n_sex_stated']

print(f'Summary shape: {summary.shape}')
print(f'Years: {sorted(summary["year"].unique())}')
print(f'ACPR regions: {summary["acpr_code"].nunique()}')
print(f'\nSample:')
cols = ['acpr_name','year','care_type','source','total_users',
        'pct_first_admission','pct_repeat_admission']
if 'pct_nesb' in summary.columns: cols.append('pct_nesb')
if 'pct_indigenous' in summary.columns: cols.append('pct_indigenous')
print(summary[cols].head(8).round(3).to_string(index=False))

In [ ]:
# =============================================================================
# STEP 4: HCP level distribution per ACPR × year
# =============================================================================
# Which HCP level is allocated most? And does the distribution differ for NESB
# vs Australian-born users? This tells us whether NESB people are accessing
# higher-intensity care or being stuck at lower levels.

if 'hcp_level' in raw.columns:
    hcp_dist = (
        raw.dropna(subset=['acpr_code', 'hcp_level'])
        .groupby(['acpr_code', 'acpr_name', 'year', 'hcp_level', 'source'])
        .agg(count=('record_count', 'sum'))
        .reset_index()
    )

    # Pivot to wide: one column per HCP level
    hcp_wide = hcp_dist.pivot_table(
        index=['acpr_code', 'acpr_name', 'year', 'source'],
        columns='hcp_level',
        values='count',
        aggfunc='sum',
        fill_value=0
    ).reset_index()
    hcp_wide.columns.name = None

    # Rename level columns to safe names
    level_rename = {}
    for col in hcp_wide.columns:
        c = str(col).strip()
        if 'Level 1' in c or c == '1': level_rename[col] = 'hcp_l1'
        elif 'Level 2' in c or c == '2': level_rename[col] = 'hcp_l2'
        elif 'Level 3' in c or c == '3': level_rename[col] = 'hcp_l3'
        elif 'Level 4' in c or c == '4': level_rename[col] = 'hcp_l4'
    hcp_wide = hcp_wide.rename(columns=level_rename)

    print('HCP distribution shape:', hcp_wide.shape)
    print('Columns:', hcp_wide.columns.tolist())
    print('\nNational HCP level totals by year:')
    level_cols = [c for c in ['hcp_l1','hcp_l2','hcp_l3','hcp_l4'] if c in hcp_wide.columns]
    nat = hcp_wide.groupby('year')[level_cols].sum()
    nat['pct_high'] = (nat.get('hcp_l3',0) + nat.get('hcp_l4',0)) / nat[level_cols].sum(axis=1) * 100
    print(nat.round(1).to_string())
else:
    hcp_wide = pd.DataFrame()
    print('hcp_level column not found')

In [ ]:
# =============================================================================
# STEP 5: Save
# =============================================================================
# Main summary: merge HCP distribution into the summary table
if not hcp_wide.empty:
    merge_cols = [c for c in ['acpr_code','year','source'] if c in hcp_wide.columns]
    summary = summary.merge(hcp_wide, on=merge_cols, how='left')

summary.to_csv(OUT, index=False)
print(f'Saved demographics_acpr.csv: {summary.shape}')
print(f'Years: {sorted(summary["year"].unique())}')
print(f'ACPR regions: {summary["acpr_code"].nunique()}')
print(f'\nColumns:')
for c in summary.columns:
    non_null = summary[c].notna().sum()
    print(f'  {c}: {non_null}/{len(summary)} non-null')